# Demo 5: KV transfer of the task span: where the goal heads read the sub-goal from

**Question.** At the pivot the goal heads encode the next sub-goal (Demo 2), and the model needs them to implement it
(Demo 4). Where do they *get* it from? The hypothesis: at the pivot they re-read the **task span**, the numbered
list of sub-tasks in the instruction. If that is right, letting the goal heads read the keys and values of a
*different* prompt's task span should change what the model writes next, and nothing else should need to change.

**Setup.** Two prompts that share the prefilled progress (`a`, `b`) and differ only in sub-task (3):

```
p_src:  (1) a  (2) b  (3) c      -> the model writes c     k/v donor
p_dst:  (1) a  (2) b  (3) d      -> the model writes d     the patched run
```

**Intervention (pivot-onwards KV transfer).** In the DST forward, from the pivot `c =` on, the **250 goal heads** read
SRC's keys / values over the task span instead of DST's own, with the transplanted values × 4. Every other head and
every other position is untouched. If the model now writes **c** in DST, the goal heads alone carried the sub-goal
over from the task span. Control: the same transfer through 250 random heads.

**Paper.** §5.1.1 (`fig:kv-transfer`), App. `app:transfer:kv` (`fig:kv-barcharts`, `tab:kv-grid`).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Non-Recurrent Goal Seeking (NRGS): at each step, re-read the goal from the task span, compare it with progress, act (§5.1).](figures/05_nrgs.png)

*Non-Recurrent Goal Seeking (NRGS): at each step, re-read the goal from the task span, compare it with progress, act (§5.1).*

![NRGS's two predictions, and the notation used in the transfer demos (prefilled sub-tasks, the run's next sub-task).](figures/05_nrgs_predictions.png)

*NRGS's two predictions, and the notation used in the transfer demos (prefilled sub-tasks, the run's next sub-task).*

![KV transfer: swap only the goal heads' view of the task span, from the pivot on (§5.1.1).](figures/05_kv_transfer_setup.png)

*KV transfer: swap only the goal heads' view of the task span, from the pivot on (§5.1.1).*

![Which states, which heads, when: one GQA group, the goal heads' keys/values over the task span (App. I.1).](figures/05_kv_transfer_detail.png)

*Which states, which heads, when: one GQA group, the goal heads' keys/values over the task span (App. I.1).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
Defaults are those of `scripts/nrgs/kv_transfer_eval.py`, the paper's main configuration: `--setup multi --span task
--goal-heads G250 --readout-heads G250 --i3-v-scale 4`, instances from `sample_multi_instances(seed=1123)`,
greedy 64 tokens, C++ library note in the system prompt (`CPP_NOTE_IN_SYSTEM = True`). GoalScope readouts use
additive_norm α 0.1, layers 35–45.

**Why this pair.** `int`, Python→Python, shared `(1) is_positive (2) count_set_bits`, SRC `(3) last_digit` vs DST
`(3) is_palindrome`. The two sub-goals have nothing in common (`n % 10` vs `str(n) == str(n)[::-1]`), so a flip is
unambiguous. In the reported run this instance is a causal HIT with a valid, shifted readout under G250, and a miss
under the random-250 control (29 of the 32 `int` Python→Python instances show that full pattern).

In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE, SRC_LANG, DST_LANG = 'int', 'python', 'python'
INST_KEY = 'multi|int|shared=is_positive-count_set_bits|src=last_digit|dst=is_palindrome'
SPAN     = 'task'      # the transplanted window: (1) .. the user turn's <|im_end|>
V_SCALE  = 4.0         # x the transplanted values read by the goal heads (paper: V x 4)
MAX_NEW  = 64

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
`src.nrgs.functional.setup` loads the model (bf16, eager) and returns the run context the NRGS helpers take.

In [3]:
import json, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
import src.nrgs.eval as E
from src.utils import env_utils

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)
DATA.CPP_NOTE_IN_SYSTEM = True                     # as in kv_transfer_eval.main()

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal_heads, ctrl_heads = load(G250), load(C250)
ctx.readout_heads, ctx.gsp = goal_heads, None
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-10-30 21:24:40 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
`sample_multi_instances` reproduces the eval's draw for this (datatype, language pair), and we pick our instance by
key. `build_geometry` builds both prompts, locates the task span in each and pads the shorter one with `\n` tokens
right before its span (`align_kv_spans`), so the two windows are position-aligned token for token.

In [4]:
insts, sample_seed = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=32, seed=1123)
inst = next(i for i in insts if PP.inst_key(i) == INST_KEY)
g = PP.build_geometry(inst, span=SPAN, ctx=ctx)

t = Table(title=f'{escape(INST_KEY)}  (sample seed {sample_seed})')
t.add_column('#'); t.add_column('SRC task list'); t.add_column('DST task list')
for k, (ps, pd) in enumerate(zip(inst._preds_for(inst.src_frontier), inst._preds_for(inst.dst_frontier)), 1):
    same = ps.id == pd.id
    t.add_row(str(k), escape(ps.description) if same else f'[bold cyan]{escape(ps.description)}[/]',
              escape(pd.description) if same else f'[bold magenta]{escape(pd.description)}[/]')
console.print(t)
print(f'window: {len(g["dst_w"])} tokens, SRC [{g["src_w"][0]}..{g["src_w"][-1]}] / DST [{g["dst_w"][0]}..{g["dst_w"][-1]}]; '
      f'padding: SRC +{g["pad_src"]} / DST +{g["pad_dst"]} newline tokens')

        multi|int|shared=is_positive-count_set_bits|src=last_digit|dst=is_palindrome  (sample seed 451643126)         
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ SRC task list                                       ┃ DST task list                                            ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ whether the number is positive                      │ whether the number is positive                           │
│ 2 │ the number of set bits in the binary representation │ the number of set bits in the binary representation      │
│ 3 │ the last decimal digit                              │ whether the number reads the same forwards and backwards │
└───┴─────────────────────────────────────────────────────┴──────────────────────────────────────────────────────────┘

window: 35 tokens, SRC [61..95] / DST [61..95]; padding: SRC +5 / DST +0 newline tokens


The two token sequences as the model sees them (user turn onward). The **task span** that is transplanted is
highlighted. Only the DST prompt runs; the goal heads read SRC's k/v over the highlighted window.

In [5]:
def show(tok, win, title, color):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        txt.append(s, style=f'bold {color} on #f1f5f9' if i in win else 'dim')
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), 'SRC prompt (k/v donor)', 'cyan')
show(g['dst_tok'], set(g['dst_w']), 'DST prompt (the run we patch)', 'magenta')

────────────────────────────────────────────────────────── SRC prompt (k/v donor) ──────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:





(1) whether the number is positive
(2) the number of set bits in the binary representation
(3) the last decimal digit
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = (n > 0)
    b = bin(abs(n)).count("1")
    c =

────────────────────────────────────────────────────── DST prompt (the run we patch) ───────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given an integer `n`, computes:
(1) whether the number is positive
(2) the number of set bits in the binary representation
(3) whether the number reads the same forwards and backwards
<|im_end|>
<|im_start|>assistant
<code>
def func(n: int):
    a = (n > 0)
    b = bin(abs(n)).count("1")
    c =

## 2. Clean runs
`PP.clean_runs` greedily continues both prompts and takes a GoalScope readout at each pivot (the goal heads' write at
`c =`, verbalized through the "What are you doing at this line of the code?" probe, as in Demo 2).

In [6]:
clean = PP.clean_runs(g, readout_heads=goal_heads, max_new=MAX_NEW, ctx=ctx)
for k in ('src', 'dst'):
    console.print(f'[bold]{k.upper()}[/] continuation: [green]{escape(clean[f"gen_{k}"].strip().splitlines()[0])}[/]')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=132 ' =' => I'm determining the last digit of the number by using the modulus operator (`% 10`).

pos=132 ' =' => I'm checking if the variable is equal to its reverse (i.e., it's a palindrome).

SRC continuation: abs(n) % 10

DST continuation: (str(n) == str(n)[::-1])

## 3. Patched run: KV transfer through the 250 goal heads (V × 4)
`PP.run_patched` = `intervention_stack` (cache SRC's keys before the rotary and its values over the window in the
goal heads' layers, then `headwise_kv_patch` rotates the keys for the DST positions and, at every step, has the goal
heads alone read them through the block's own attention call, `self_attn.interface`) around a greedy decode, plus a
GoalScope readout at the patched pivot. We also run the V × 1 arm (no value boost; reported as a side arm) and the
random-250 control (V × 4; the paper ran the control without readouts, we add one for illustration).

In [7]:
arms = {
    'goal 250, V x4 (paper)':  dict(heads=goal_heads, v=V_SCALE),
    'goal 250, V x1':          dict(heads=goal_heads, v=1.0),
    'control 250, V x4':       dict(heads=ctrl_heads, v=V_SCALE),
}
pat = {name: PP.run_patched(g, goal_heads=a['heads'], readout_heads=goal_heads, v_scale=a['v'], max_new=MAX_NEW, ctx=ctx)
       for name, a in arms.items()}

pos=132 ' =' => I'm looking at the last digit of the number by using the modulus operator (`% 10`).

pos=132 ' =' => I'm checking the last digit of the number by using the modulus operator (`% 10`).

pos=132 ' =' => I'm checking if the variable is equal to its reverse (i.e., it's a palindrome).

## 4. Judge
`PP.judge` is the batch eval's oracle. **causal** = does the patched continuation's *first* computation implement
SRC's sub-goal (c)? (`judge_causal`; a BOTH verdict counts as a miss.) **readout** = are the clean readouts valid
(SRC→c, DST→d), and did the patched readout shift to c? (`eval_readouts`.) Claude calls are cached on disk.

In [8]:
verd = {}
for name, p in pat.items():
    raw = dict(src_prompt=g['src_prompt'], dst_prompt=g['dst_prompt'], **clean, **p)
    verd[name] = PP.judge(inst, raw)

first = lambda s: s.strip().splitlines()[0] if s.strip() else ''
lab = lambda l: {'src': '[bold cyan]c (SRC)[/]', 'dst': '[bold magenta]d (DST)[/]'}.get(l, f'[red]{l}[/]')
t = Table(title=f'SRC sub-goal c = "{inst.expected_next_desc}"   |   DST sub-goal d = "{inst.dst_next_desc}"', show_lines=True)
t.add_column('run'); t.add_column('continuation of `c =`', max_width=46); t.add_column('implements')
t.add_column('GoalScope readout at the pivot', max_width=52); t.add_column('readout says')
rd0 = verd['goal 250, V x4 (paper)'][1]
t.add_row('clean SRC', escape(first(clean['gen_src'])), '-', escape(clean['ro_src'].strip()), lab(rd0['source']))
t.add_row('clean DST', escape(first(clean['gen_dst'])), '-', escape(clean['ro_dst'].strip()), lab(rd0['dest']))
for name, p in pat.items():
    c, r = verd[name]
    hit = '[green]HIT[/]' if c['triggered'] else '[red]miss[/]'
    t.add_row(f'patched: {name}', escape(first(p['gen_patched'])), f"{lab(c['label'])} {hit}",
              escape(p['ro_patched'].strip()), lab(r['patched']) + ('  [green]shifted[/]' if r['shifted'] else ''))
console.print(t)

        SRC sub-goal c = "the last decimal digit"   |   DST sub-goal d = "whether the number reads the same forwards and backwards"         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ run                             ┃ continuation of `c =`    ┃ implements   ┃ GoalScope readout at the pivot            ┃ readout says     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ clean SRC                       │ abs(n) % 10              │ -            │ I'm determining the last digit of the     │ c (SRC)          │
│                                 │                          │              │ number by using the modulus operator (`%  │                  │
│                                 │                          │              │ 10`).                                     │                  │
├─────────────────────────────────┼──────────────────────────┼──────────────┼───────────────────────────────────────────┼──────────────────┤
│ clean DST                       │ (str(n) == str(n)[::-1]) │ -            │ I'm checking if the variable is equal to  │ d (DST)          │
│                                 │                          │              │ its reverse (i.e., it's a palindrome).    │                  │
├─────────────────────────────────┼──────────────────────────┼──────────────┼───────────────────────────────────────────┼──────────────────┤
│ patched: goal 250, V x4 (paper) │ abs(n) % 10              │ c (SRC) HIT  │ I'm looking at the last digit of the      │ c (SRC)  shifted │
│                                 │                          │              │ number by using the modulus operator (`%  │                  │
│                                 │                          │              │ 10`).                                     │                  │
├─────────────────────────────────┼──────────────────────────┼──────────────┼───────────────────────────────────────────┼──────────────────┤
│ patched: goal 250, V x1         │ (abs(n) % 10)            │ c (SRC) HIT  │ I'm checking the last digit of the number │ c (SRC)  shifted │
│                                 │                          │              │ by using the modulus operator (`% 10`).   │                  │
├─────────────────────────────────┼──────────────────────────┼──────────────┼───────────────────────────────────────────┼──────────────────┤
│ patched: control 250, V x4      │ (str(n) == str(n)[::-1]) │ d (DST) miss │ I'm checking if the variable is equal to  │ d (DST)          │
│                                 │                          │              │ its reverse (i.e., it's a palindrome).    │                  │
└─────────────────────────────────┴──────────────────────────┴──────────────┴───────────────────────────────────────────┴──────────────────┘

How to read it. Only the goal heads' read of the task span changed, and DST now writes SRC's sub-goal
(`abs(n) % 10`, the last digit) instead of its own palindrome check. The GoalScope readout at the patched pivot
moves with it: the plan changed before any token was written. The same transfer through 250 random heads changes
nothing. Without the value boost (V × 1) this instance still flips, but pooled over all 512 instances V × 1 flips
only 72% against 94% at V × 4 (table below).

## 5. Cross-check against the reported run
The cached records of this instance (`results/diff/<model>/path_patch/<run_tag>/int__python-python/results.json`):
same greedy decodes, same verdicts (up to bf16 nondeterminism on other hardware). Then the reported rates for this
cell and pooled over all 16 cells (4 datatypes × 4 language pairs, n = 512).

In [9]:
R = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', MODEL.split('/')[-1], 'path_patch')
TAGS = {'goal 250, V x4 (paper)': 'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250',
        'goal 250, V x1':         'I1-off__I2-off__I3-on__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250',
        'control 250, V x4':      'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-control250_250__gh-control250_250__ro-off'}
cell = f'{PP._safe(DATATYPE)}__{SRC_LANG}-{DST_LANG}'
t = Table(title='this notebook vs. the reported run (same instance)')
t.add_column('run'); t.add_column('paper causal'); t.add_column('ours'); t.add_column('same patched gen?')
t.add_column(f'{cell} causal rate'); t.add_column('pooled causal (n)'); t.add_column('pooled readout shift')
for name, tag in TAGS.items():
    rec = next(r for r in json.load(open(os.path.join(R, tag, cell, 'results.json'))) if r['key'] == INST_KEY)
    meta = json.load(open(os.path.join(R, tag, cell, 'meta.json')))
    metas = [json.load(open(os.path.join(R, tag, d, 'meta.json'))) for d in sorted(os.listdir(os.path.join(R, tag)))
             if os.path.exists(os.path.join(R, tag, d, 'meta.json'))]
    n = sum(m['n_instances'] for m in metas)
    pc = sum(m['causal_trigger_rate'] * m['n_instances'] for m in metas) / n
    rs = [m for m in metas if m.get('readout_shift_rate') is not None]
    pr = (f"{sum(m['readout_shift_rate'] * m['n_instances'] for m in rs) / sum(m['n_instances'] for m in rs):.3f}" if rs else 'off')
    t.add_row(name, 'HIT' if rec['causal']['triggered'] else 'miss', 'HIT' if verd[name][0]['triggered'] else 'miss',
              str(rec['raw']['gen_patched'] == pat[name]['gen_patched']), f"{meta['causal_trigger_rate']:.3f}",
              f'{pc:.3f} ({n})', pr)
console.print(t)

                                             this notebook vs. the reported run (same instance)                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃                        ┃              ┃      ┃                   ┃ int__python-python causal  ┃                   ┃                      ┃
┃ run                    ┃ paper causal ┃ ours ┃ same patched gen? ┃ rate                       ┃ pooled causal (n) ┃ pooled readout shift ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ goal 250, V x4 (paper) │ HIT          │ HIT  │ True              │ 0.969                      │ 0.936 (512)       │ 0.900                │
│ goal 250, V x1         │ HIT          │ HIT  │ True              │ 0.750                      │ 0.721 (512)       │ 0.670                │
│ control 250, V x4      │ miss         │ miss │ True              │ 0.000                      │ 0.002 (512)       │ off                  │
└────────────────────────┴──────────────┴──────┴───────────────────┴────────────────────────────┴───────────────────┴──────────────────────┘

## Where this goes in the paper / how to reproduce at scale
- **Commands**: `job_list/diff_eval/path_patch_32b_listgap.json` (16 jobs = 4 datatypes × 4 language pairs), e.g.
  ```
  python scripts/nrgs/kv_transfer_eval.py --datatypes int --lang-transfers py-py --n-samples 32 --i3-v-scale 1 2 4
  ```
  and the random-head control, the `ppctrl_*` jobs of `job_list/diff_eval/path_patch_control_32b_listgap.json`
  (`--goal-heads C250 --readout-heads C250 --no-readout`). Smoke: `--max-instances 2`. Records are resumable and land in
  `results/diff/Qwen2.5-Coder-32B-Instruct/path_patch/<run_tag>/<dt>__<src>-<dst>/`; point `RESULTS_DIR` in
  `env.yml` to a scratch dir for a re-run.
- **Paper numbers**: pooled n = 512, causal 0.936, readout shift 0.900 (≈94% / ≈90%); random-250 control 1/512.
- **Generator**: `notebooks/fig_n_table_generators/kv_transfer.ipynb` (cells 7 / 11 / 13: `kv_barcharts`,
  the right panel of `kv-transfer-main`, `tab:kv-grid`).
- **Related**: Q transfer with gaze-select-restrict (`scripts/nrgs/q_transfer_eval.py`, §5.1.2) and task span vs
  code-progress span (`scripts/nrgs/task_vs_progress_eval.py`, §5.2).